## Floodfill implementation

In [ ]:
def get_neighborhood(x, y, z, shape):
    x, y, z = int(round(x)), int(round(y)), int(round(z))
    neighbors = []
    for i in range(-1, 2):
        for j in range(-1, 2):
            for k in range(-1, 2):
                if i + j + k == 0:
                    continue
                nx, ny, nz = x + i, y + j, z + k
                if 0 <= nz < shape[0] and 0 <= ny < shape[1] and 0 <= nx < shape[2]:
                    neighbors.append((nx, ny, nz))
    return neighbors

In [ ]:
from collections import deque
import numpy as np


def floodfill_opt(img, initial_mask, max_voxels=None, forbidden_mask=None, alpha=0.8):
    """Flood-fills a spine region across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Args:
        img (array): 3D image stack (Z, Y, X).
        initial_mask (array): 3D binary mask for current spine.
        max_voxels (int): optional cap on max number of voxels to include.
        forbidden_mask (array): 3D binary mask where fill is not allowed (e.g. dendrite).
        alpha (float): parameter that determines the cut on intensity for a voxel to be chosen as seed.
                       Default to 0.8.

    Returns:
        array: 3D binary mask after flood-fill.
    """
    final_mask = np.zeros_like(img, dtype=bool)
    final_mask[initial_mask] = True

    # Use a high-intensity voxel from the mask as seed
    zyx_coords = np.argwhere(initial_mask)
    seed_z, seed_y, seed_x = zyx_coords[np.argmax(img[initial_mask])]
    seed_intensity = float(img[seed_z, seed_y, seed_x])
    threshold = min(seed_intensity * alpha,
                    np.percentile(img[initial_mask], 85))

    # Initialize queue
    queue = deque()
    queue.extend(get_neighborhood(seed_x, seed_y, seed_z, img.shape))

    # Loop over neightbors
    while queue:
        x, y, z = queue.popleft()

        # Bounds check
        if not (0 <= z < img.shape[0] and 0 <= y < img.shape[1] and 0 <= x < img.shape[2]):
            continue

        # Already visited or forbidden voxels are discarded
        if final_mask[z, y, x]:
            continue
        if forbidden_mask is not None and forbidden_mask[z, y, x]:
            continue

        # Intensity check. If intense enough, the neightbor is added to the mask
        if img[z, y, x] > threshold:
            final_mask[z, y, x] = True
            neighbors = get_neighborhood(x, y, z, img.shape)
            for nx, ny, nz in neighbors:
                if forbidden_mask is not None and forbidden_mask[nz, ny, nx]:
                    continue
                if final_mask[nz, ny, nx]:
                    continue
                queue.append((nx, ny, nz))

            if max_voxels is not None and np.count_nonzero(final_mask) > max_voxels:
                print(
                    f"Floodfill aborted: number of voxels added ({np.count_nonzero(final_mask)}) exceeded max_voxels ({max_voxels})"
                )
                break

    return final_mask

In [ ]:
import numpy as np
import imageio as io
import matplotlib.pyplot as plt
from skimage.measure import label

img_folder = "test_images"
for animal in {"turtles"}:  # , "mice"}:
    stack = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_orig.tif"))
    spines = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_spines.tif"))
    dendrite = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_dendrite.tif"))

    # Label spines
    L = label(spines)

    # Get all spine labels (exclude background = 0)
    labels = np.unique(L)
    labels = labels[labels != 0]

    # Final mask to accumulate filled results
    final_mask = np.zeros_like(L, dtype=bool)

    # Optional: store diffs to analyze floodfill additions
    diff_map = np.zeros_like(L, dtype=np.int8)

    # Get dendrite mask and extend it over the whole Z axis for avoiding floodfilling spines into dendrite
    dendrite_mask = dendrite == 255
    dendrite_mask_combined = np.any(dendrite_mask, axis=0)
    dendrite_mask_broad = np.broadcast_to(
        dendrite_mask_combined, dendrite_mask.shape)

    # Loop through all spines
    for label_id in labels:
        mask = L == label_id
        filled = floodfill_opt(stack, mask, forbidden_mask=dendrite_mask_broad)

        # Accumulate into final mask
        final_mask |= filled

        # Optional: difference map
        diff = filled.astype(int) - mask.astype(int)
        diff_map += diff.astype(np.int8)  # accumulate all diffs

    io.mimwrite(f"{img_folder}/out/{animal}_spines_flooded.tif",
                final_mask.astype(np.uint8) * 255)

    # Save or visualize final mask
    plt.figure()
    plt.imshow(final_mask.max(0), cmap='gray')
    plt.title(f"Final {animal} accumulated mask (max projection)")
    plt.axis('off')
    plt.show()

    # Show total diff (accumulated)
    plt.figure()
    plt.imshow(diff_map.max(0), cmap='bwr', vmin=-1, vmax=1)
    plt.title(f"Total {animal} difference map: red = added, blue = lost")
    plt.axis('off')
    plt.show()

    # Show final labels compared with new labels
    L_flood = label(final_mask)
    plt.subplot(121)
    plt.imshow(L.max(0), cmap='viridis')
    plt.title(f"Initial {animal} masks")
    plt.axis('off')

    plt.subplot(122)
    plt.imshow(L_flood.max(0), cmap='viridis')
    plt.title(f"Final {animal} masks")
    plt.axis('off')

## Show overlay for comparing new and old labels

In [ ]:
for label_id in labels:
    old_mask = (L == label_id)

    # Find the new labels that overlap with the old ones (ignorinig bg)
    overlapping_new_labels = np.unique(L_flood[old_mask])
    print(f"Label {label_id} now is {overlapping_new_labels}")
    overlapping_new_labels = overlapping_new_labels[overlapping_new_labels != 0]

    new_mask = np.isin(L_flood, overlapping_new_labels)

    old_mask_proj = old_mask.max(axis=0)
    new_mask_proj = new_mask.max(axis=0)
    diff = new_mask_proj & ~old_mask_proj

    # Plot overlay for comparison: red for old and blue for new
    overlay = np.zeros(old_mask_proj.shape + (3,), dtype=np.uint8)
    overlay[..., 0] = old_mask_proj * 255
    overlay[..., 2] = diff * 255
    plt.imshow(overlay)
    ys, xs = np.where(old_mask_proj)
    plt.xlim(xs.min()-100, xs.max()+100)
    plt.ylim(ys.max()+100, ys.min()-100)
    plt.title(
        f"Comparison for label {label_id} (red = original, blue = new with floodfill)")
    plt.axis('off')
    plt.show()

# Plot general overlay
# * red = original
# * blue = added masks with floodfill
# * pink/purple-ish = re-labeled masks with floodfill
L_proj = (L > 0).max(axis=0)
diff_proj = diff_map.max(axis=0)
overlay = np.zeros(L_proj.shape + (3,), dtype=np.uint8)
overlay[..., 0] = L_proj * 255
overlay[..., 2] = diff_proj * 255

plt.figure()
plt.imshow(overlay)
plt.title("General overview")
plt.axis('off')
plt.show()